# Stage 4b — Data augmentation and ensembling (making the CNN and the GRU generalize better)

**Why this notebook.** Every model so far overfits: training accuracy reaches ~100 % while validation stays at 55–66 %, and the gap between
signers seen and unseen in training grows with model flexibility (+0.5, +2.9, +4.4, +7.6 points). Both symptoms say the same thing:
the networks memorize the few clips they have (about 17 per sign) and the style of the signers who recorded them.
**Augmentation** shows the network a slightly different version of every clip at every epoch (tilted, faster, noisier...), so it cannot
memorize exact numbers and must rely on what stays the same: the hand shape and movement. **Ensembling** averages several models.
(This is the planned "Stage 6", done before the Transformer so that every later model is compared fairly.)

**Bars to beat** (5-fold CV, 300 signs, chance 0.33 %): Stage 3 CNN **63.6 ± 4.0 %**, Stage 4 GRU **60.9 ± 4.0 %**;
leakage gaps (seen minus unseen signers): CNN +4.4, GRU +7.6 points.

**The augmentations (each applied to a clip with probability 0.5, with a random strength).** They act on the *normalized hand shapes*,
before the per-channel standardization, so that a rotation is a real rotation:
| Name | What it simulates |
|---|---|
| `rotation` | the hand is tilted in the image (+/- 15 degrees at strength 1) |
| `scale` | different hand proportions (+/- 15 %) |
| `timewarp` | the signer is faster or slower, and the clip starts a bit earlier or later (+/- 20 % speed, +/- 4 frames) |
| `framehold` | frames without a detection (10 % of frames repeat the previous frame; the real data lacks about 7 % of frames) |
| `noise` | small landmark jitter (0.03 hand units, visible hands only) |
| `jointdrop` | random landmark glitches (5 % of joints set to 0) |
| `wristjitter` | the hand is held a bit higher/lower/left/right in the picture (0.05) |
Mirroring (swapping the hands) is *not* tested: handedness can carry meaning in sign languages.

**What the notebook does**
1. Unit-tests the augmentations and plots them on a real clip (Cells 5 and 6).
2. Part A (CNN, fast): tunes the recipe on **validation only** with 2 seeds: each augmentation alone, then a greedy combination, then the strength.
3. Trains the CNN with the recipe: 3 seeds, 5-fold CV (with and without augmentation, same settings), leakage re-test.
4. Part B (GRU): the same recipe on the Stage 4 GRU: 3 seeds, 5-fold CV, leakage re-test.
5. Ensembles: several seeds, and CNN + GRU, including a 5-fold CV figure.

**How to run:** a new Colab notebook with a **GPU runtime**; run the cells in order from Cell 2.
Needs in Drive: `splits.csv`, `landmarks_raw.npz`, `frames_tracked_v2.npy` (Stage 0) and, if present, the `results/stage*_results.csv` files for the paired comparisons.
Rough time (an estimate, not measured): Part A about 25 minutes, CNN finals about 15 minutes, Part B about 40 minutes.
Set `RUN_GRU = False` in Cell 2 to do only the CNN part first. Results are saved after every phase.
If a cell fails, send me the last lines of the error.

In [ ]:
# [setup] Cell 2 — configuration
from pathlib import Path
import json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/slovo_project')
except ImportError:                      # running outside Colab
    BASE = Path('./slovo_project')

PROC, FIG, RES = BASE / 'processed', BASE / 'figures', BASE / 'results'
for d in (FIG, RES):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
FIXED_T = 64
N_FOLDS = 5
SWEEP_SEEDS = [0, 1]          # two seeds per configuration while tuning
FINAL_SEEDS = [0, 1, 2]
RUN_GRU = True                # Part B (about 40 minutes)
RUN_NOAUG_CV = True           # also cross-validate the CNN WITHOUT augmentation under identical settings (fair paired comparison)
warnings.filterwarnings('ignore')
print('torch', torch.__version__, '| device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'no GPU')

In [ ]:
# Cell 3 — load the Stage 0 outputs (same files and same split as Stages 1 to 4)
splits = pd.read_csv(PROC / 'splits.csv')
z = np.load(PROC / 'landmarks_raw.npz', allow_pickle=False)
ids, lengths = z['ids'], z['lengths']
offsets = np.concatenate([[0], np.cumsum(lengths)])
id2idx = {c: i for i, c in enumerate(ids)}
frames = np.load(PROC / 'frames_tracked_v2.npy', mmap_mode='r')
assert len(frames) == offsets[-1], 'frames_tracked_v2.npy does not match landmarks_raw.npz: re-run Stage 0 cells 10 and 11b.'

meta = splits[splits['in_subset']].reset_index(drop=True)
y_all = meta['label_id_subset'].astype(int).to_numpy()
N_CLASSES = int(y_all.max()) + 1
assert len(np.unique(y_all)) == N_CLASSES

tr = (meta['split'] == 'train').to_numpy(); va = (meta['split'] == 'val').to_numpy(); te = (meta['split'] == 'test').to_numpy()
folds = meta['fold'].to_numpy()
print(f'{len(meta)} clips | {N_CLASSES} signs | chance {1 / N_CLASSES:.2%} | train {tr.sum()} val {va.sum()} test {te.sum()}')

In [ ]:
# [sequence] Cell 4 — the raw sequences: 64 frames x 132 numbers per clip (cached by Stage 2; rebuilt if missing)
from tqdm.auto import tqdm
WRIST, MIDDLE_MCP = 0, 9

def normalize_hands(x, aspect=None, eps=1e-6):
    x = x.astype(np.float32).copy()
    if aspect is not None:
        x[..., 0] *= aspect
    centred = x - x[..., WRIST:WRIST + 1, :]
    scale = np.linalg.norm(centred[..., MIDDLE_MCP, :2], axis=-1)
    scale = np.where(scale < eps, np.nan, scale)
    return centred / scale[..., None, None]

def clip_to_sequence(clip, aspect, T=FIXED_T):
    """(L, 2, 21, 3) -> (T, 132): 126 normalized joint values (hand 0 then hand 1, joint-major x,y,z), 4 wrist-position
    values, 2 'hand visible' flags. Missing hands are filled with 0."""
    L = len(clip)
    idx = np.linspace(0, L - 1, T).round().astype(int)
    hands = normalize_hands(clip, aspect=aspect)[idx]
    wrist = clip[:, :, WRIST, :2].astype(np.float32).copy()
    wrist[..., 0] *= aspect
    wrist = wrist[idx]
    present = ~np.isnan(hands).all(axis=(2, 3))
    hands = np.where(np.isnan(hands), 0.0, hands)
    wrist = np.where(present[:, :, None], wrist, 0.0)
    return np.concatenate([hands.reshape(T, 126), wrist.reshape(T, 4), present.astype(np.float32)], axis=1).astype(np.float32)

SEQF = PROC / f'sequences_subset_T{FIXED_T}.npz'
meta_ids = meta['attachment_id'].to_numpy().astype(str)
SEQ = None
if SEQF.exists():
    zs = np.load(SEQF, allow_pickle=False)
    if zs['ids'].shape == meta_ids.shape and (zs['ids'] == meta_ids).all():
        SEQ = zs['S'].astype(np.float32)
        print('Loaded cached sequences:', SEQF.name)
if SEQ is None:
    SEQ = np.zeros((len(meta), FIXED_T, 132), np.float32)
    for r, (cid, asp) in enumerate(tqdm(zip(meta['attachment_id'], meta['aspect']), total=len(meta))):
        i = id2idx[cid]
        SEQ[r] = clip_to_sequence(np.asarray(frames[offsets[i]:offsets[i + 1]], dtype=np.float32), float(asp))
    np.savez_compressed(SEQF, S=SEQ.astype(np.float16), ids=meta_ids)
    print('Saved', SEQF.name)
assert SEQ.shape == (len(meta), FIXED_T, 132) and np.isfinite(SEQ).all()
print('sequence tensor:', SEQ.shape)

In [ ]:
# [aug] Cell 5 — the augmentations (numpy, applied to RAW normalized hand shapes) and their unit tests
def seq_mask(S):
    """(N, T, 132) mask: 1 where the channel's hand is visible (or the channel is a visibility flag)."""
    p = S[:, :, 130:132]
    return np.concatenate([np.repeat(p[:, :, :1], 63, 2), np.repeat(p[:, :, 1:], 63, 2),
                           np.repeat(p[:, :, :1], 2, 2), np.repeat(p[:, :, 1:], 2, 2), np.ones_like(p)], axis=2)

def seq_stats(S_tr):
    m = seq_mask(S_tr)
    cnt = np.maximum(m.sum((0, 1)), 1.0)
    mean = (S_tr * m).sum((0, 1)) / cnt
    std = np.sqrt((((S_tr - mean) ** 2) * m).sum((0, 1)) / cnt)
    std = np.where(std < 1e-3, 1.0, std)
    mean[130:], std[130:] = 0.0, 1.0
    return mean.astype(np.float32), std.astype(np.float32)

def apply_seq_norm(S, mean, std):
    return (((S - mean) / std) * seq_mask(S)).astype(np.float32)           # missing hands stay exactly 0

BASE_MAG = {'rotation': 15.0, 'scale': 0.15, 'timewarp': 0.20, 'noise': 0.03,
            'framehold': 0.10, 'jointdrop': 0.05, 'wristjitter': 0.05}
ORDER = ['timewarp', 'framehold', 'rotation', 'scale', 'noise', 'jointdrop', 'wristjitter']
P_APPLY = 0.5                                                              # each augmentation is applied to a clip with this probability

def _joints(x):
    return x[:, :, :126].reshape(len(x), x.shape[1], 2, 21, 3)

def _put(x, j):
    out = x.copy()
    out[:, :, :126] = j.reshape(len(x), x.shape[1], 126)
    return out

def _on(rng, B):
    return (rng.random(B) < P_APPLY).astype(np.float32)

def aug_rotation(x, rng, mag):                                             # mag = maximum angle in degrees
    B = len(x)
    th = np.deg2rad(rng.uniform(-mag, mag, B)) * _on(rng, B)
    c = np.cos(th).reshape(B, 1, 1, 1).astype(np.float32); s = np.sin(th).reshape(B, 1, 1, 1).astype(np.float32)
    j = _joints(x).copy(); xx, yy = j[..., 0].copy(), j[..., 1].copy()
    j[..., 0] = c * xx - s * yy
    j[..., 1] = s * xx + c * yy
    return _put(x, j)

def aug_scale(x, rng, mag):                                                # mag = maximum relative change
    B = len(x)
    f = (1 + rng.uniform(-mag, mag, B) * _on(rng, B)).astype(np.float32).reshape(B, 1, 1, 1, 1)
    return _put(x, _joints(x) * f)

def aug_timewarp(x, rng, mag):                                             # mag = maximum relative speed change (shift = mag * 20 frames)
    B, T = x.shape[:2]
    on = _on(rng, B)
    r = 1 + rng.uniform(-mag, mag, B) * on
    sh = rng.uniform(-mag, mag, B) * 20 * on
    t, c = np.arange(T)[None, :], (T - 1) / 2
    idx = np.clip(np.rint(c + (t - c) * r[:, None] + sh[:, None]), 0, T - 1).astype(int)
    return np.take_along_axis(x, idx[:, :, None], axis=1)

def aug_framehold(x, rng, mag):                                            # mag = probability that a frame repeats the previous one
    B, T = x.shape[:2]
    drop = (rng.random((B, T)) < min(mag, 0.5)) & (_on(rng, B)[:, None] > 0)
    drop[:, 0] = False
    idx = np.maximum.accumulate(np.where(drop, 0, np.arange(T)[None, :]), axis=1)    # last kept frame
    return np.take_along_axis(x, idx[:, :, None], axis=1)

def aug_noise(x, rng, mag):                                                # mag = standard deviation, visible hands only
    B, T = x.shape[:2]
    vis = x[:, :, 130:132].reshape(B, T, 2, 1, 1)
    n = rng.normal(0, mag, (B, T, 2, 21, 3)).astype(np.float32) * vis * _on(rng, B).reshape(B, 1, 1, 1, 1)
    return _put(x, _joints(x) + n)

def aug_jointdrop(x, rng, mag):                                            # mag = probability that a joint is set to 0
    B, T = x.shape[:2]
    m = (rng.random((B, T, 2, 21, 1)) < min(mag, 0.5)) & (_on(rng, B).reshape(B, 1, 1, 1, 1) > 0)
    return _put(x, np.where(m, 0.0, _joints(x)).astype(np.float32))

def aug_wristjitter(x, rng, mag):                                          # mag = standard deviation of a constant shift, visible hands only
    B, T = x.shape[:2]
    vis = x[:, :, 130:132].reshape(B, T, 2, 1)
    off = rng.normal(0, mag, (B, 1, 2, 2)).astype(np.float32) * _on(rng, B).reshape(B, 1, 1, 1)
    out = x.copy()
    out[:, :, 126:130] = (x[:, :, 126:130].reshape(B, T, 2, 2) + off * vis).reshape(B, T, 4)
    return out

AUG_FUNCS = {'rotation': aug_rotation, 'scale': aug_scale, 'timewarp': aug_timewarp, 'noise': aug_noise,
             'framehold': aug_framehold, 'jointdrop': aug_jointdrop, 'wristjitter': aug_wristjitter}

def augment(x, names, rng, strength=1.0):
    """Apply the named augmentations (always in the same order) to a batch x of RAW sequences (B, T, 132)."""
    for name in ORDER:
        if name in names:
            x = AUG_FUNCS[name](x, rng, BASE_MAG[name] * strength)
    return x

# --- unit tests -------------------------------------------------------------------------------------------------------
_rng = np.random.default_rng(0)
_x = SEQ[tr][:32].copy()

def _consistent(o):
    """Wherever a hand's visibility flag is 0, that hand's joints and wrist position must be exactly 0."""
    B, T = o.shape[:2]
    inv = 1.0 - o[:, :, 130:132]
    return (np.abs(o[:, :, :126].reshape(B, T, 2, 63)) * inv[..., None]).max() == 0 and \
           (np.abs(o[:, :, 126:130].reshape(B, T, 2, 2)) * inv[..., None]).max() == 0

assert _consistent(_x), 'the input data itself is inconsistent'
for name in ORDER:
    out = augment(_x, (name,), _rng)
    assert out.shape == _x.shape and out.dtype == np.float32 and np.isfinite(out).all(), name
    assert _consistent(out), f'{name}: a hand that is flagged missing has non-zero values'
    if name not in ('timewarp', 'framehold'):                                  # these two only reorder frames in time
        assert np.array_equal(out[:, :, 130:132], _x[:, :, 130:132]), f'{name} changed the visibility flags'
    assert not np.array_equal(out, _x) or name in ('timewarp', 'framehold'), f'{name} did nothing'
    zero = AUG_FUNCS[name](_x, _rng, 0.0)                                      # magnitude 0 = identity
    assert np.allclose(zero, _x, atol=1e-6), f'{name} with magnitude 0 is not the identity'
# rotation keeps distances; framehold repeats the previous frame; timewarp at speed 1 and shift 0 is the identity
P_APPLY_BACKUP, P_APPLY = P_APPLY, 1.0
r = aug_rotation(_x, _rng, 30.0)
assert np.allclose(np.linalg.norm(_joints(r)[..., :2], axis=-1), np.linalg.norm(_joints(_x)[..., :2], axis=-1), atol=1e-4), 'rotation changed distances'
h = aug_framehold(_x, _rng, 0.3)
for b in range(len(h)):
    for t in range(1, h.shape[1]):
        assert np.array_equal(h[b, t], _x[b, t]) or np.array_equal(h[b, t], h[b, t - 1]), 'framehold produced a frame that is neither new nor repeated'
P_APPLY = P_APPLY_BACKUP
print('OK: augmentation unit tests passed for', ORDER)

In [ ]:
# Cell 6 — look at the augmentations on a real clip (fingertip x-coordinate over time; thin lines = 5 random draws)
Xs = SEQ[tr]
cands = np.where(Xs[:, :, 130].mean(1) > 0.9)[0]                              # clips where hand 0 is visible in >90 % of the frames
assert len(cands) > 0, 'no clip with a visible hand 0'
k = cands[np.argmax(Xs[cands][:, :, 24].std(1))]                              # the one whose fingertip moves the most (a flat line teaches nothing)
clip = Xs[k:k + 1].copy()
P_APPLY_BACKUP, P_APPLY = P_APPLY, 1.0                                       # force the augmentation on, just for the plot
fig, axes = plt.subplots(2, 3, figsize=(13, 5.5), sharex=True)
for ax, name in zip(axes.ravel(), ['timewarp', 'framehold', 'rotation', 'scale', 'noise', 'jointdrop']):
    for r in range(5):
        a = augment(clip, (name,), np.random.default_rng(r), strength=1.0)
        ax.plot(a[0, :, 24], lw=0.8, alpha=0.8)                              # channel 24 = x of joint 8 (index fingertip) of hand 0
    ax.plot(clip[0, :, 24], 'k', lw=2, label='original'); ax.set_title(name); ax.legend(fontsize=7)
plt.suptitle('x-coordinate of the index fingertip of hand 0 (normalized hand units) after each augmentation'); plt.tight_layout()
plt.savefig(FIG / 'stage4b_augmentations.png', dpi=150); plt.show()
P_APPLY = P_APPLY_BACKUP

In [ ]:
# [torch] Cell 7 — the two networks (CNN and GRU, both take (batch, time, channels)) and the training loop with augmentation
import torch.nn as nn
import torch.nn.functional as F

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class CNN1D(nn.Module):
    """Stage 3 network: blocks of [Conv1d -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout], global average + max pooling, Linear."""
    def __init__(self, in_ch, n_classes, channels=(128, 128, 256), kernel=9, dropout=0.5, block_dropout=0.1):
        super().__init__()
        blocks, c_in = [], in_ch
        for c_out in channels:
            blocks += [nn.Conv1d(c_in, c_out, kernel, padding=kernel // 2), nn.BatchNorm1d(c_out), nn.ReLU(),
                       nn.MaxPool1d(2), nn.Dropout(block_dropout)]
            c_in = c_out
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * c_in, n_classes))

    def forward(self, x):                                         # x: (batch, time, channels)
        h = self.features(x.transpose(1, 2))
        h = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)
        return self.head(h)

class GRUNet(nn.Module):
    """Stage 4 network: optional conv front-end, (Bi)GRU, pooling over time, Linear. Input (batch, time, channels)."""
    def __init__(self, in_ch, n_classes, hidden=256, layers=2, bidirectional=True, dropout=0.5, rnn_dropout=0.3,
                 pooling='meanmax', frontend='conv', conv_ch=128):
        super().__init__()
        if frontend == 'conv':
            self.frontend = nn.Sequential(nn.Conv1d(in_ch, conv_ch, 5, padding=2), nn.BatchNorm1d(conv_ch), nn.ReLU())
            rnn_in = conv_ch
        else:
            self.frontend, rnn_in = None, in_ch
        self.rnn = nn.GRU(rnn_in, hidden, num_layers=layers, batch_first=True, bidirectional=bidirectional,
                          dropout=rnn_dropout if layers > 1 else 0.0)
        d = hidden * (2 if bidirectional else 1)
        self.pooling, self.bidirectional = pooling, bidirectional
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * d if pooling == 'meanmax' else d, n_classes))

    def forward(self, x):
        if self.frontend is not None:
            x = self.frontend(x.transpose(1, 2)).transpose(1, 2)
        out, h_n = self.rnn(x)
        if self.pooling == 'meanmax':
            h = torch.cat([out.mean(dim=1), out.amax(dim=1)], dim=1)
        elif self.bidirectional:
            h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h = h_n[-1]
        return self.head(h)

def new_model(cfg, in_ch, seed):
    set_seed(seed)
    if cfg['kind'] == 'cnn':
        m = CNN1D(in_ch, N_CLASSES, cfg['channels'], cfg['kernel'], cfg['dropout'], cfg['block_dropout'])
    else:
        m = GRUNet(in_ch, N_CLASSES, cfg['hidden'], cfg['layers'], cfg['bidirectional'], cfg['dropout'], cfg['rnn_dropout'],
                   cfg['pooling'], cfg['frontend'], cfg['conv_ch'])
    return m.to(DEVICE)

@torch.no_grad()
def check_architecture(cfg, in_ch=132):
    """Dummy batch through the network: returns (output shape, number of parameters)."""
    m = new_model(cfg, in_ch, 0).eval()
    out = m(torch.zeros(2, FIXED_T, in_ch, device=DEVICE))
    return tuple(out.shape), sum(p.numel() for p in m.parameters())

@torch.no_grad()
def _eval(model, X, y):
    model.eval()
    logits = model(X)
    return F.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

def fit(model, Xtr_raw, ytr, mean, std, Xva=None, yva=None, aug=(), strength=1.0, epochs=200, lr=1e-3, weight_decay=1e-2,
        batch_size=128, patience=25, label_smoothing=0.1, seed=0, fixed_epochs=None, grad_clip=0.0):
    """Mini-batch training as in Stages 2 to 4. Xtr_raw are the RAW training sequences (N, T, 132); when `aug` is not empty,
    every mini-batch is augmented on the fly (numpy) and then standardized with the TRAIN mean/std. Xva is already standardized."""
    t0 = time.time()
    n_epochs = fixed_epochs or epochs
    g = torch.Generator(); g.manual_seed(seed)                              # order of the mini-batches
    arng = np.random.default_rng(10_000 + seed)                             # randomness of the augmentation
    Xclean = torch.as_tensor(apply_seq_norm(Xtr_raw, mean, std), dtype=torch.float32, device=DEVICE)   # for the training-accuracy curve
    yt = torch.as_tensor(ytr, dtype=torch.long, device=DEVICE)
    use_val = (Xva is not None) and (fixed_epochs is None)
    if use_val:
        Xv = torch.as_tensor(Xva, dtype=torch.float32, device=DEVICE)
        yv = torch.as_tensor(yva, dtype=torch.long, device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_epoch, best_state, bad, n = -1.0, 0, None, 0, len(Xtr_raw)
    for epoch in range(1, n_epochs + 1):
        model.train()
        perm = torch.randperm(n, generator=g)                              # CPU tensor
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            if aug:
                xb = torch.as_tensor(apply_seq_norm(augment(Xtr_raw[idx.numpy()], aug, arng, strength), mean, std),
                                     dtype=torch.float32, device=DEVICE)
            else:
                xb = Xclean[idx.to(DEVICE)]
            yb = yt[idx.to(DEVICE)]
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
        tl, ta = _eval(model, Xclean, yt)
        hist['train_loss'].append(tl); hist['train_acc'].append(ta)
        if use_val:
            vl, vacc = _eval(model, Xv, yv)
            hist['val_loss'].append(vl); hist['val_acc'].append(vacc)
            if vacc > best_acc:
                best_acc, best_epoch, bad = vacc, epoch, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
    if use_val and best_state is not None:
        model.load_state_dict(best_state)
    else:
        best_epoch = n_epochs
    return hist, best_epoch, time.time() - t0

@torch.no_grad()
def predict_logits(model, X, batch_size=512):
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        xb = torch.as_tensor(X[i:i + batch_size], dtype=torch.float32, device=DEVICE)
        out.append(model(xb).cpu().numpy())
    return np.concatenate(out)

print('networks and training loop defined')

In [ ]:
# [runner] Cell 8 — configurations, metrics and the experiment runner
from sklearn.metrics import f1_score

CNN_CFG = dict(kind='cnn', channels=(128, 128, 256), kernel=9, dropout=0.5, block_dropout=0.1, grad_clip=0.0)   # Stage 3 final
GRU_CFG = dict(kind='gru', hidden=256, layers=2, bidirectional=True, dropout=0.5, rnn_dropout=0.3, pooling='meanmax',
               frontend='conv', conv_ch=128, grad_clip=1.0)                                                      # Stage 4 final
TRAIN_CFG = dict(lr=1e-3, weight_decay=1e-2, batch_size=128, epochs=200, patience=25, label_smoothing=0.1)      # more room than before: augmentation converges slower

for nm, cfg in (('CNN', CNN_CFG), ('GRU', GRU_CFG)):
    shape, n_par = check_architecture(cfg)
    assert shape == (2, N_CLASSES), f'unexpected output shape {shape}'
    print(f'{nm}: output {shape} | {n_par:,} parameters')

def metrics_from_logits(logits, y):
    pred = logits.argmax(1)
    top5 = np.argsort(-logits, axis=1)[:, :5]
    return {'n': int(len(y)), 'top1': float((pred == y).mean()), 'top5': float(np.mean([yt in t for yt, t in zip(y, top5)])),
            'macro_f1': float(f1_score(y, pred, average='macro')), 'pred': pred, 'logits': logits.astype(np.float32)}

def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

MASKS = {'train': tr, 'val': va, 'test': te}
results = []
_cache = {}

def mask_key(masks):
    return tuple((k, hash(m.tobytes())) for k, m in masks.items())

def run_one(experiment, name, model_cfg, aug, strength, masks, seed, fixed_epochs=None, use_cache=True):
    """Train one model on masks['train'] (augmenting if `aug` is not empty), evaluate on every other mask."""
    ckey = (json.dumps(model_cfg, sort_keys=True, default=list), tuple(aug), float(strength), seed, mask_key(masks), fixed_epochs)
    if use_cache and ckey in _cache:
        return _cache[ckey]
    Xtr = SEQ[masks['train']]
    mean, std = seq_stats(Xtr)
    ev = {k: apply_seq_norm(SEQ[m], mean, std) for k, m in masks.items() if k != 'train'}
    model = new_model(model_cfg, 132, seed)
    hist, best_ep, fit_s = fit(model, Xtr, y_all[masks['train']], mean, std, ev.get('val'), y_all[masks['val']] if 'val' in masks else None,
                               aug=tuple(aug), strength=strength, seed=seed, fixed_epochs=fixed_epochs,
                               grad_clip=model_cfg['grad_clip'], **TRAIN_CFG)
    out = {'name': name, 'seed': seed, 'hist': hist, 'best_epoch': best_ep, 'fit_s': fit_s, 'metrics': {}}
    for sname, m in masks.items():
        if sname == 'train':
            continue
        r = metrics_from_logits(predict_logits(model, ev[sname]), y_all[m])
        out['metrics'][sname] = r
        results.append({'experiment': experiment, 'name': name, 'aug': '+'.join(aug) or 'none', 'strength': strength, 'seed': seed,
                        'set': sname, 'n': r['n'], 'top1': r['top1'], 'top5': r['top5'], 'macro_f1': r['macro_f1'],
                        'best_epoch': best_ep, 'fit_s': fit_s})
    del model
    if use_cache:
        _cache[ckey] = out
    return out

def val_of(outs):
    v = [o['metrics']['val']['top1'] for o in outs]
    return float(np.mean(v)), float(np.std(v))

def summary_line(o, sets=('val', 'test')):
    return ' | '.join(f'{s} top-1 {o["metrics"][s]["top1"]:.1%} top-5 {o["metrics"][s]["top5"]:.1%}' for s in sets if s in o['metrics']) \
           + f' | best epoch {o["best_epoch"]} | {o["fit_s"]:.0f}s'

def save_results(tag=''):
    pd.DataFrame(results).to_csv(RES / 'stage4b_results.csv', index=False)

In [ ]:
# Cell 9 — sanity check: the augmented training pipeline must still be able to memorize 64 clips (if not, something is wrong)
Xtr = SEQ[tr]; mean, std = seq_stats(Xtr)
small = np.random.default_rng(0).choice(len(Xtr), 64, replace=False)
cfg0 = {**CNN_CFG, 'dropout': 0.0, 'block_dropout': 0.0}
mdl = new_model(cfg0, 132, 0)
h, _, _ = fit(mdl, Xtr[small], y_all[tr][small], mean, std, aug=('noise', 'rotation'), strength=0.5, fixed_epochs=200,
              lr=3e-3, weight_decay=0.0, batch_size=64, label_smoothing=0.0, seed=0)
print(f'train loss {h["train_loss"][0]:.2f} -> {h["train_loss"][-1]:.3f} | clean train accuracy {h["train_acc"][0]:.0%} -> {h["train_acc"][-1]:.0%}')
assert h['train_acc'][-1] > 0.8, 'The CNN cannot memorize 64 clips with mild augmentation: something is wrong (send me this output).'
print('OK: the training pipeline works')
del mdl, Xtr

In [ ]:
# Cell 10 — Part A, step 1: the CNN without augmentation, then each augmentation alone (validation only, 2 seeds)
t_start = time.time()
base = [run_one('A0 no augmentation', 'CNN no aug', CNN_CFG, (), 1.0, MASKS, s) for s in SWEEP_SEEDS]
base_val, base_std = val_of(base)
print(f'no augmentation                 val top-1 {base_val:.1%} +/- {base_std:.1%} | best epochs {[o["best_epoch"] for o in base]} | {np.mean([o["fit_s"] for o in base]):.0f}s per run')

single = {}
for name in ORDER:
    outs = [run_one('A1 single augmentation', f'CNN + {name}', CNN_CFG, (name,), 1.0, MASKS, s) for s in SWEEP_SEEDS]
    single[name] = val_of(outs)[0]
    print(f'{name:16s} alone          val top-1 {single[name]:.1%} +/- {val_of(outs)[1]:.1%} | gain {single[name] - base_val:+.1%} | best epochs {[o["best_epoch"] for o in outs]}')
save_results()
print(f'\nstep 1 took {(time.time() - t_start) / 60:.0f} min')

plt.figure(figsize=(7, 3.2))
gains = [single[n] - base_val for n in ORDER]
plt.bar(ORDER, gains, color=['#4C78A8' if g > 0 else '#999' for g in gains]); plt.axhline(0, color='k', lw=0.8)
plt.ylabel('validation top-1 gain vs no augmentation'); plt.xticks(rotation=25, ha='right'); plt.title('Each augmentation alone (CNN, mean of 2 seeds)')
plt.tight_layout(); plt.savefig(FIG / 'stage4b_single_gains.png', dpi=150); plt.show()

In [ ]:
# Cell 11 — Part A, step 2: combine the augmentations greedily, then tune the overall strength (validation only)
ranked = sorted(single, key=lambda n: -single[n])
recipe, best_val = [], base_val
for name in ranked:
    cand = tuple(sorted(recipe + [name], key=ORDER.index))
    outs = [run_one('A2 combination', 'CNN + ' + '+'.join(cand), CNN_CFG, cand, 1.0, MASKS, s) for s in SWEEP_SEEDS]
    v, sd = val_of(outs)
    keep = v > best_val + 0.005                                           # keep an augmentation only if validation improves by more than 0.5 point
    print(f'{"+".join(cand):60s} val top-1 {v:.1%} +/- {sd:.1%} -> {"kept" if keep else "dropped"}')
    if keep:
        recipe, best_val = list(cand), v
recipe = tuple(sorted(recipe, key=ORDER.index))
print('\nchosen combination:', recipe if recipe else 'none (no augmentation helped on validation)')

best_strength, best_sv = 1.0, best_val
if recipe:
    print('\nstrength of the combination (1.0 = the magnitudes in the table at the top):')
    for m in (0.5, 1.0, 2.0):
        outs = [run_one('A3 strength', f'CNN + recipe x{m}', CNN_CFG, recipe, m, MASKS, s) for s in SWEEP_SEEDS]
        v, sd = val_of(outs)
        print(f'  strength x{m:<4} val top-1 {v:.1%} +/- {sd:.1%} | best epochs {[o["best_epoch"] for o in outs]}')
        if v > best_sv + 0.0001:
            best_strength, best_sv = m, v
print(f'\nFINAL RECIPE: {recipe}, strength x{best_strength}  (validation {best_sv:.1%} vs {base_val:.1%} without augmentation)')
json.dump({'recipe': list(recipe), 'strength': best_strength, 'val_no_aug': base_val, 'val_aug': best_sv}, open(RES / 'stage4b_recipe.json', 'w'))
save_results()
print(f'part A took {(time.time() - t_start) / 60:.0f} min in total')

In [ ]:
# Cell 12 — the CNN with the recipe: 3 seeds, with and without augmentation (identical settings), and the seed ensemble
def softmax_ensemble(outs, sname):
    P = np.mean([softmax(o['metrics'][sname]['logits']) for o in outs], axis=0)
    return metrics_from_logits(np.log(P + 1e-9), y_all[MASKS[sname]])

fin = {}
for tag, aug in (('CNN no aug', ()), ('CNN + aug', recipe)):
    fin[tag] = [run_one('F1 CNN seeds', tag, CNN_CFG, aug, best_strength, MASKS, s, use_cache=False) for s in FINAL_SEEDS]
    v = [o['metrics']['val']['top1'] for o in fin[tag]]; t = [o['metrics']['test']['top1'] for o in fin[tag]]; t5 = [o['metrics']['test']['top5'] for o in fin[tag]]
    e = softmax_ensemble(fin[tag], 'test')
    print(f'{tag:11s} val {np.mean(v):.1%} +/- {np.std(v):.1%} | test {np.mean(t):.1%} +/- {np.std(t):.1%} | test top-5 {np.mean(t5):.1%} | ensemble of 3: test top-1 {e["top1"]:.1%} top-5 {e["top5"]:.1%}')
save_results()

In [ ]:
# Cell 13 — the CNN, 5-fold cross-validation (same folds as Stages 1 to 4), with and without augmentation
def cv_run(tag, model_cfg, aug, strength, experiment):
    acc, logits, ys = [], [], []
    for k in range(N_FOLDS):
        te_k = folds == k; va_k = folds == (k + 1) % N_FOLDS; tr_k = ~(te_k | va_k)
        o = run_one(experiment, tag, model_cfg, aug, strength, {'train': tr_k, 'val': va_k, 'test': te_k}, 0, use_cache=False)
        results[-2]['set'], results[-1]['set'] = f'fold {k} val', f'fold {k}'
        acc.append(o['metrics']['test']['top1']); logits.append(o['metrics']['test']['logits']); ys.append(y_all[te_k])
        print(f'  {tag} fold {k}: test clips {te_k.sum()} | top-1 {acc[-1]:.1%} | best epoch {o["best_epoch"]} | {o["fit_s"]:.0f}s')
    acc = np.array(acc)
    print(f'{tag}: top-1 mean {acc.mean():.1%} +/- {acc.std():.1%} over folds | pooled {np.mean(np.concatenate([l.argmax(1) for l in logits]) == np.concatenate(ys)):.1%}\n')
    save_results()
    return {'acc': acc, 'logits': logits, 'y': ys}

CV = {}
CV['CNN + aug'] = cv_run('CNN + aug', CNN_CFG, recipe, best_strength, 'F2 CNN 5-fold CV')
if RUN_NOAUG_CV:
    CV['CNN no aug'] = cv_run('CNN no aug', CNN_CFG, (), best_strength, 'F2 CNN 5-fold CV')

def cv_from_csv(path, experiment, name):
    if not path.exists():
        return None
    d = pd.read_csv(path)
    col = 'name' if 'name' in d else 'model'
    d = d[(d.experiment == experiment) & (d[col] == name) & d.set.str.fullmatch(r'fold \d')]
    v = d.sort_values('set').top1.to_numpy()
    return v if len(v) == N_FOLDS else None

REFS = {'Stage 1 linear SVM': cv_from_csv(RES / 'stage1_results.csv', '5-fold CV', 'linear SVM'),
        'Stage 2 MLP on features': cv_from_csv(RES / 'stage2_results.csv', 'E5 5-fold CV', 'MLP on features'),
        'Stage 3 CNN (no aug)': cv_from_csv(RES / 'stage3_results.csv', 'E4 5-fold CV', 'CNN tuned'),
        'Stage 4 GRU (no aug)': cv_from_csv(RES / 'stage4_results.csv', 'E4 5-fold CV', 'GRU tuned')}

def compare(tag, acc, others):
    for nm, v in others.items():
        if v is None:
            continue
        diff = acc - v
        print(f'  {tag} vs {nm:24s} {v.mean():.1%} +/- {v.std():.1%} | better in {(diff > 0).sum()} of {N_FOLDS} folds | mean difference {diff.mean():+.1%} '
              f'(std error {diff.std(ddof=1) / np.sqrt(N_FOLDS):.1%}; per fold: ' + ', '.join(f'{x:+.1%}' for x in diff) + ')')

others = {**REFS}
if 'CNN no aug' in CV:
    others['CNN no aug (this notebook)'] = CV['CNN no aug']['acc']
print('Paired comparison of the augmented CNN:')
compare('CNN + aug', CV['CNN + aug']['acc'], others)

In [ ]:
# Cell 14 — the CNN: signer-leakage re-test (official split, seen vs unseen signers), with and without augmentation
off_train = meta['official_train'].to_numpy(bool)
unseen = meta['official_test_unseen_signer'].to_numpy(bool)
seen = (~off_train) & (~unseen)
print(f'official train {off_train.sum()} | official test: {seen.sum()} clips from seen signers, {unseen.sum()} from unseen signers')

def leakage(tag, model_cfg, aug, strength, epochs, experiment):
    runs = [run_one(experiment, tag, model_cfg, aug, strength, {'train': off_train, 'seen': seen, 'unseen': unseen}, s,
                    fixed_epochs=epochs, use_cache=False) for s in FINAL_SEEDS]
    a = [o['metrics']['seen']['top1'] for o in runs]; b = [o['metrics']['unseen']['top1'] for o in runs]
    print(f'{tag:11s} (fixed {epochs} epochs) seen {np.mean(a):.1%} +/- {np.std(a):.1%} | unseen {np.mean(b):.1%} +/- {np.std(b):.1%} | gap {np.mean(a) - np.mean(b):+.1%}')
    save_results()
    return np.mean(a) - np.mean(b), np.mean(b)

LEAK = {}
for tag, aug in (('CNN no aug', ()), ('CNN + aug', recipe)):
    ep = int(np.median([o['best_epoch'] for o in fin[tag]]))
    LEAK[tag] = leakage(tag, CNN_CFG, aug, best_strength, ep, 'F3 CNN leakage')
print('(Stage 3 CNN without augmentation: seen 61.6%, unseen 57.3%, gap +4.4; the standard error of a gap is about 2.7 points)')

In [ ]:
# Cell 15 — Part B: the same recipe on the Stage 4 GRU (3 seeds, 5-fold CV, leakage)
if RUN_GRU:
    t_gru = time.time()
    fin['GRU + aug'] = [run_one('G1 GRU seeds', 'GRU + aug', GRU_CFG, recipe, best_strength, MASKS, s, use_cache=False) for s in FINAL_SEEDS]
    v = [o['metrics']['val']['top1'] for o in fin['GRU + aug']]; t = [o['metrics']['test']['top1'] for o in fin['GRU + aug']]
    e = softmax_ensemble(fin['GRU + aug'], 'test')
    print(f'GRU + aug  val {np.mean(v):.1%} +/- {np.std(v):.1%} | test {np.mean(t):.1%} +/- {np.std(t):.1%} | ensemble of 3: test top-1 {e["top1"]:.1%} top-5 {e["top5"]:.1%}'
          f'   (Stage 4 GRU without augmentation: val 60.4, test 63.7 +/- 0.9)')
    save_results()
    CV['GRU + aug'] = cv_run('GRU + aug', GRU_CFG, recipe, best_strength, 'G2 GRU 5-fold CV')
    print('Paired comparison of the augmented GRU:')
    compare('GRU + aug', CV['GRU + aug']['acc'], {**REFS, 'CNN + aug (this notebook)': CV['CNN + aug']['acc']})
    ep = int(np.median([o['best_epoch'] for o in fin['GRU + aug']]))
    LEAK['GRU + aug'] = leakage('GRU + aug', GRU_CFG, recipe, best_strength, ep, 'G3 GRU leakage')
    print('(Stage 4 GRU without augmentation: seen 57.7%, unseen 50.1%, gap +7.6)')
    print(f'part B took {(time.time() - t_gru) / 60:.0f} min')
else:
    print('RUN_GRU is False: skipped')

In [ ]:
# Cell 16 — ensembles: several seeds, and different architectures (CNN + GRU), with a 5-fold CV figure
print('single split (test fold), ensemble = average of the predicted probabilities:')
for combo in [('CNN + aug',), ('GRU + aug',), ('CNN + aug', 'GRU + aug')]:
    if not all(c in fin for c in combo):
        continue
    outs = [o for c in combo for o in fin[c]]
    for sname in ('val', 'test'):
        e = softmax_ensemble(outs, sname)
        single = np.mean([o['metrics'][sname]['top1'] for o in outs])
        print(f'  {" + ".join(combo):22s} {len(outs)} models {sname:4s} top-1 {e["top1"]:.1%} top-5 {e["top5"]:.1%} | average single model {single:.1%} | gain {e["top1"] - single:+.1%}')

if 'GRU + aug' in CV:
    ens_acc = []
    for k in range(N_FOLDS):
        P = (softmax(CV['CNN + aug']['logits'][k]) + softmax(CV['GRU + aug']['logits'][k])) / 2
        ens_acc.append(float((P.argmax(1) == CV['CNN + aug']['y'][k]).mean()))
    CV['CNN + GRU ensemble'] = {'acc': np.array(ens_acc)}
    print(f'\n5-fold CV, ensemble of the augmented CNN and GRU (one model of each per fold): {np.mean(ens_acc):.1%} +/- {np.std(ens_acc):.1%}')
    compare('CNN + GRU ensemble', CV['CNN + GRU ensemble']['acc'], {'Stage 3 CNN (no aug)': REFS['Stage 3 CNN (no aug)'], 'CNN + aug (this notebook)': CV['CNN + aug']['acc']})

In [ ]:
# Cell 17 — results table and the final figure
res = pd.DataFrame(results)
res.to_csv(RES / 'stage4b_results.csv', index=False)
print(res[res.experiment.str.startswith(('F', 'G'))].groupby(['experiment', 'name', 'set'])[['top1', 'top5', 'macro_f1']].mean().round(3).to_string())

labels, means, stds, colors = [], [], [], []
for nm, v in REFS.items():
    if v is not None:
        labels.append(nm); means.append(v.mean()); stds.append(v.std()); colors.append('#999')
for nm in ('CNN + aug', 'GRU + aug', 'CNN + GRU ensemble'):
    if nm in CV:
        labels.append('Stage 4b: ' + nm); means.append(CV[nm]['acc'].mean()); stds.append(CV[nm]['acc'].std()); colors.append('#4C78A8')
plt.figure(figsize=(10, 4))
plt.bar(range(len(labels)), means, yerr=stds, capsize=4, color=colors)
plt.xticks(range(len(labels)), labels, rotation=25, ha='right'); plt.ylabel('top-1 accuracy (5-fold CV, mean +/- std)')
plt.axhline(1 / N_CLASSES, color='k', ls='--', lw=1); plt.title('Effect of augmentation and ensembling')
plt.tight_layout(); plt.savefig(FIG / 'stage4b_results.png', dpi=150); plt.show()
print('Saved', RES / 'stage4b_results.csv', RES / 'stage4b_recipe.json', 'and', FIG / 'stage4b_results.png')

## Stage 4b checklist

- [ ] Cell 5 printed `OK: augmentation unit tests passed` and Cell 6 shows the augmented curves
- [ ] Cell 9 printed `OK: the training pipeline works`
- [ ] Cell 10: the single-augmentation table and bar chart
- [ ] Cell 11: the chosen recipe and strength
- [ ] Cells 12 to 14: the CNN with 3 seeds, 5-fold CV and the paired comparisons, leakage with and without augmentation
- [ ] Cell 15: the GRU (if `RUN_GRU = True`)
- [ ] Cell 16: ensembles
- [ ] `results/stage4b_results.csv`, `results/stage4b_recipe.json` and the figures exist

**Send back:** the printed outputs of Cells 5 and 9 to 16, and the figures `stage4b_augmentations.png`, `stage4b_single_gains.png` and `stage4b_results.png`.
If a cell fails, send me the last lines of the error.